In [ ]:
!pip install -qqq torchmetrics

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as scheduler
import torch.utils.data as data

from torchvision.io import read_image
from torchmetrics.functional import structural_similarity_index_measure as ssim

import os
import random

In [ ]:
class Normalize(nn.Module):
    def __init__(self, upper_val = 255.0):
        super().__init__()
        self.upper_val = upper_val

    @torch.no_grad()
    def forward(self, image: torch.tensor, target: torch.tensor):
        return (
            image.float() / self.upper_val,
            target.float() / self.upper_val
        )
    
    def __repr__(self):
        return f"Divide all pixels in image by: {self.upper_val}"

In [ ]:
class RotateImage(nn.Module):
    def __init__(self, prob=0.25, dims=(-2, -1)):
        super().__init__()
        self.prob = prob
        self.dims = dims

    @torch.no_grad()
    def forward(self, image, target):
        if random.random() > self.prob:
            return image, target

        return (
            torch.rot90(image, 1, self.dims),
            torch.rot90(target, 1, self.dims),
        )
    
    def __repr__(self):
        return f"Rotate image with around dims: {self.dims} probability: {self.prob}"

In [ ]:
class RollImage(nn.Module):
    def __init__(self, prob=0.25):
        super().__init__()
        self.prob = prob

    @torch.no_grad()
    def forward(self, image, target):
        if random.random() > self.prob:
            return image, target
        
        shifts=random.randint(1, 128)
        return (
            torch.roll(image, shifts=shifts),
            torch.roll(target, shifts=shifts),
        )
    
    def __repr__(self):
        return f"Roll image with shifts: {shifts}"

In [ ]:
class Transforms(nn.Module):
    def __init__(self, modules: list[nn.Module] = [
        RotateImage(),
        RollImage(),
        Normalize()
    ]):
        super().__init__()
        self.transforms = nn.ModuleList([*modules])

    def forward(self, image, target):
        for module in self.transforms:
            image, target = module(image, target)

        return image, target

In [ ]:
class ImageDataset(data.Dataset):
    def __init__(self, split, transform=None):
        super().__init__()
        self.split = split
        self.image_dir = f"/kaggle/input/phasedatagen/data/{self.split}/images"
        self.targets_dir = f"//kaggle/input/phasedatagen/data/{self.split}/labels"
        self.image_files = sorted(os.listdir(self.image_dir))
        self.transform = transform

    def __getitem__(self, index):
        fname = self.image_files[index]
        noisy_img = read_image(f"{self.image_dir}/{fname}")
        tname = fname.replace("image", "target")
        target_img = read_image(f"{self.targets_dir}/{tname}")
        if self.transform is not None:
            return self.transform(noisy_img, target_img)

        return noisy_img, target_img

    def __len__(self):
        return len(os.listdir(f"/kaggle/input/phasedatagen/data/{self.split}/images"))    

In [ ]:
TrainDataset = ImageDataset("train", transform=Transforms())
ValDataset = ImageDataset("val", transform=Transforms())
TestDataset = ImageDataset("test", transform=Transforms([Normalize()]))

In [ ]:
TrainLoader = data.DataLoader(
    TrainDataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
ValLoader = data.DataLoader(
    ValDataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
TestLoader = data.DataLoader(
    TestDataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
class CBAMChannelAttn(nn.Module):
    def __init__(self, channels, image_dim, r):
        super().__init__()
        self.image_dim = image_dim
        self.channels = channels

        self.mlp = nn.Sequential(
            nn.Linear(in_features=channels, out_features=int(channels/r)),
            nn.Linear(in_features=int(channels/r), out_features=channels)
        )
        self.maxpool = nn.MaxPool2d(image_dim)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, input):
        avgr = self.mlp(self.avgpool(input).squeeze(-2, -1))
        maxr = self.mlp(self.maxpool(input).squeeze(-2, -1))
        return F.sigmoid(avgr+maxr).unsqueeze(-1).unsqueeze(-1)

class CBAMSpatialAttn(nn.Module):
    def __init__(self, channels, image_dim):
        super().__init__()
        self.image_dim = image_dim
        self.channels = channels

        self.avgpool = nn.AdaptiveAvgPool3d((1,image_dim,image_dim))
        self.maxpool = nn.MaxPool3d((channels,1,1))
        self.conv = nn.Conv2d(in_channels=2, out_channels=1, kernel_size=7, padding=3)

    def forward(self, input):
        maxr = self.maxpool(input)
        avgr = self.avgpool(input)
        concatd = torch.cat((maxr, avgr), dim=-3)
        return F.sigmoid(self.conv(concatd))

class CBAMAttn(nn.Module):
    def __init__(self, channels, image_dim, r):
        super().__init__()
        self.channel_attn = CBAMChannelAttn(channels, image_dim, r)
        self.spatial_attn = CBAMSpatialAttn(channels, image_dim)

    def forward(self, input):
        channel_attn_out = input * self.channel_attn(input)
        return channel_attn_out * self.spatial_attn(channel_attn_out)

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(
        self, 
        image_dim: int, 
        in_channels: int, 
        out_channels: int, 
        kernel_size: int, 
        non_linearity: nn.Module,
        downsample: bool = True
    ):
        super().__init__()
        self.image_dim = image_dim
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size

        self.conv1 = nn.Conv2d(
            in_channels=self.in_channels, 
            out_channels=self.out_channels, 
            kernel_size=kernel_size, 
            padding=2
        )
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.non_linearity = non_linearity()
        self.bn = nn.BatchNorm2d(self.out_channels)
        self.conv2 = nn.Conv2d(
            in_channels=self.out_channels, 
            out_channels=self.out_channels, 
            kernel_size=kernel_size, 
            padding=2
        )
        self.cbam_attn = CBAMAttn(
            channels=self.out_channels,
            image_dim=int(self.image_dim/2),
            r=1.25
        )

        if downsample:
            self.identity = nn.Sequential(
                self.pool,
                nn.Conv2d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=self.kernel_size,
                    padding=2
                )
            )
        else:
            self.identity = nn.Identity()

    def forward(self, x: torch.tensor):
        if x.shape[-1] != self.image_dim and x.shape[-2] != self.image_dim:
            raise ValueError(f"Tensor of invalid size: {x.shape} passed into Encoder block")

        xres = self.identity(x)
        xout1 = self.bn(self.non_linearity(self.conv1(x)))
        xout2 = self.pool(self.non_linearity(self.conv2(xout1)))
        return self.cbam_attn(xout2 + xres)

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(
        self, 
        image_dim: int, 
        in_channels: int, 
        out_channels: int, 
        kernel_size: int, 
        non_linearity: nn.Module,
        upsample: bool = True
    ):
        super().__init__()
        self.image_dim = image_dim
        self.in_channels = in_channels
        self.out_channels = out_channels 
        self.kernel_size = kernel_size

        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv1 = nn.Conv2d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            padding=2
        )
        self.non_linearity = non_linearity()
        self.bn = nn.BatchNorm2d(self.out_channels)
        self.conv2 = nn.Conv2d(
            in_channels=self.out_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            padding=2
        )
        self.cbam_attn = CBAMAttn(
            channels=self.out_channels,
            image_dim=self.image_dim*2,
            r=1.25
        )

        if upsample:
            self.identity = nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                nn.Conv2d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=self.kernel_size,
                    padding=2
                )
            )
        else:
            self.identity = nn.Identity()

    def forward(self, x: torch.tensor, skip_channel: torch.tensor = None):
        if x.shape[-1] != self.image_dim or x.shape[-2] != self.image_dim:
            raise ValueError(f"x with invalid shape: {x.shape} provided, expected last 2 dimensions: {self.image_dim}")

        if skip_channel is not None:
            if skip_channel.shape[-1] != self.image_dim or skip_channel.shape[-2] != self.image_dim:
                raise ValueError(f"Skip channel with invalid shape: {skip_channel.shape} provided, expected last 2 dimensions: {self.image_dim}")

            try:
                assert x.shape == skip_channel.shape
            except AssertionError:
                print("Shapes of input tensor and skip channel given to Decoder block are not same.")
                print(f"Shape of x: {x.shape}, Shape of skip channel: {skip_channel.shape}")

            x = torch.cat((x, skip_channel), axis=1)
            
        xres = self.identity(x)
        x = self.upsample(x)
        xout1 = self.bn(self.non_linearity(self.conv1(x)))
        xout2 = self.non_linearity(self.conv2(xout1))
        return self.cbam_attn(xout2 + xres)

In [ ]:
class UNet(nn.Module):
    def __init__(
        self, 
        input_dim: int, 
        output_dim: int, 
        channels: list[int],
        skip_channels: bool = True
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.channels = channels

        self.encoder = nn.ModuleList()
        current_img_dim = self.input_dim
        prev_channel = 1
        for i in range(len(channels)):
            self.encoder.append(
                EncoderBlock(
                    image_dim=int(current_img_dim),
                    in_channels=prev_channel,
                    out_channels=channels[i],
                    kernel_size=5,
                    non_linearity=nn.ReLU,
                    downsample=True
                )
            )
            current_img_dim /= 2
            prev_channel = channels[i]

        self.decoder = nn.ModuleList()
        channels.reverse()
        channels = channels[1:]
        channels.append(1)
        for i in range(len(channels)):
            self.decoder.append(
                DecoderBlock(
                    image_dim=int(current_img_dim),
                    in_channels=prev_channel if not skip_channels else prev_channel*2,
                    out_channels=channels[i],
                    kernel_size=5,
                    non_linearity=nn.ReLU,
                    upsample=True
                )
            )
            current_img_dim *= 2
            prev_channel = channels[i]

        del prev_channel, current_img_dim

        self.final_conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=1)

    def forward(self, x: torch.tensor):
        if x.shape[-1] != self.input_dim or x.shape[-2] != self.input_dim:
            raise ValueError(f"Expected last 2 dimensions of x.shape: {self.input_dim}, recieved: {x.shape}") 
        
        xout = x
        layer_outputs = []

        for encoder_block in self.encoder:
            xout = encoder_block(xout)
            layer_outputs.append(xout)

        reversed_layer_outputs = list(reversed(layer_outputs))
        for i, decoder_block in enumerate(self.decoder):
            xout = decoder_block(xout, reversed_layer_outputs[i])

        return self.final_conv(xout)

In [ ]:
os.makedirs('/kaggle/working/models/', exist_ok=True)
os.makedirs('/kaggle/working/optimizers/', exist_ok=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

num_train = len(TrainLoader)
num_val = len(ValLoader)
num_test = len(TestLoader)

MSELoss = nn.MSELoss()

In [ ]:
model1 = UNet(
    input_dim=256,
    output_dim=256,
    channels=[16, 32, 64, 128, 256]
)

modelg1 = model1.to(device=device)

total_trainable = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f"Trainable parameters in model1: {total_trainable}")

In [ ]:
optimizer1 = optim.Adam(modelg1.parameters(), lr=1e-5)
LRScheduler1 = scheduler.ReduceLROnPlateau(optimizer1, factor=0.1, patience=3)

train_mse_losses = []
val_mse_losses = []
val_snrs = []

for epoch in range(25):
    print(f" === EPOCH {epoch} === ")
    mse_loss = 0.0
    epoch_mse_train = 0.0

    modelg1.train()
    for noisy, clean in TrainLoader:
        noisy=noisy.to(device)
        clean=clean.to(device)
        modelout = modelg1(noisy)
        mse_loss = MSELoss(modelout, clean)

        optimizer1.zero_grad()
        mse_loss.backward()
        optimizer1.step()

        epoch_mse_train += mse_loss

    print(f"Train MSE loss: {epoch_mse_train / num_train}")
    train_mse_losses.append((epoch_mse_train / num_train).cpu().detach().numpy())
    
    epoch_val_mse = 0.0
    epoch_val_snr = 0.0
    modelg1.eval()
    with torch.no_grad():
        for noisy, clean in ValLoader:
            noisy=noisy.to(device)
            clean=clean.to(device)
            modelout = modelg1(noisy)
            mse = MSELoss(modelout, clean)
            epoch_val_mse += mse
            epoch_val_snr += -10.0*torch.log10(mse)
            
        print(f"Val MSE Loss: {epoch_val_mse / num_val}")
        val_mse_losses.append((epoch_val_mse / num_val).cpu().detach().numpy())
        print(f"Average PSNR over validation set: {epoch_val_snr / num_val}")
        val_snrs.append((epoch_val_snr / num_val).cpu().detach().numpy())

    print("\n")
    LRScheduler1.step(epoch_val_snr)

    if (epoch+1) % 5 == 0:
        torch.save(modelg1.state_dict(), f"/kaggle/working/models/model-epoch{epoch}.pth")
        torch.save(optimizer1.state_dict(), f"/kaggle/working/optimizers/optimizer-epoch{epoch}.pth")        

In [ ]:
import matplotlib.pyplot as plt
plt.plot(train_mse_losses)
plt.show()

In [ ]:
plt.plot(val_mse_losses)
plt.show()

In [ ]:
plt.plot(val_snrs)
plt.show()

In [ ]:
mse_loss = 0.0
epoch_test_snr = 0.0
epoch_test_loss = 0.0
ssim_loss = 0.0

modelg1.eval()
with torch.no_grad():
    for noisy, clean in TestLoader:
        noisy=noisy.to(device)
        clean=clean.to(device)
        modelout = modelg1(noisy)
        mse_loss = MSELoss(modelout, clean).item()
        epoch_test_loss += mse_loss
        epoch_test_snr += -10.0*torch.log10(mse_loss)
        ssim_loss += ssim(preds=modelout, target=clean, data_range=1.0)

print(f"Test MSE Loss: {mse_loss / num_test}")
print(f"Average SNR over test set: {epoch_test_snr / num_test}")
print(f"Average SSIM over test set: {ssim_loss / num_test}")